# 07 — Traditional ML & Ensemble (No Deep Learning)

Baseline: flatten raw pixel values from resized images, train RF/DT/SVM/LR classifiers. No CNN feature extractor — this is the pure classical ML comparison group.

## Section 0: Google Colab Setup & Dataset Download

**Run this cell first every time you open a new Colab session.**

### One-time Colab Secrets setup
Go to **Runtime → Manage secrets** and add these three secrets:

| Secret name | Value |
|---|---|
| `KAGGLE_USERNAME` | sk1285 |
| `KAGGLE_KEY` | (your kaggle API key from kaggle.com/settings/account) |
| `HF_TOKEN` | (your HuggingFace token from huggingface.co/settings/tokens) |

Once secrets are saved they persist across all Colab sessions — you only do this once.

### What this cell does
- Installs `kaggle` and `huggingface_hub`
- Reads credentials from Colab Secrets
- Downloads the Brain Tumor MRI dataset once to `/content/MRI_DATASET/`  
  (all 8 notebooks share the same folder — subsequent notebooks skip the download)
- Sets path variables used by later cells

In [ ]:
import sys, os, json

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    os.system("pip install huggingface_hub -q")
    from google.colab import drive, userdata

    HF_TOKEN = userdata.get('HF_TOKEN')   # Colab Secret: HF_TOKEN

    print("Mounting Google Drive...")
    drive.mount('/content/drive')

    # Dataset lives in Google Drive.
    # One-time setup: open the link below, click "Add shortcut to Drive",
    # place it in "My Drive" and name it exactly  MRI_DATASET
    # https://drive.google.com/drive/folders/15cP-SVH3BT20ogDjuwS5tXnVuXIW9Doe
    DATASET_PATH     = "/content/drive/MyDrive/MRI_DATASET/"
    SAVED_MODELS_DIR = "/content/saved_models/"
    RESULTS_DIR      = "/content/results/"

    if not os.path.isdir(DATASET_PATH + "Training"):
        raise RuntimeError(
            "Dataset not found at " + DATASET_PATH + "\n"
            "Fix:\n"
            "  1. Open: https://drive.google.com/drive/folders/15cP-SVH3BT20ogDjuwS5tXnVuXIW9Doe\n"
            "  2. Click 'Add shortcut to Drive' → My Drive\n"
            "  3. Name the shortcut exactly:  MRI_DATASET\n"
            "  4. Re-run this cell"
        )

else:
    HF_TOKEN     = os.environ.get('HF_TOKEN', '')
    DATASET_PATH     = "../MRI_DATASET/"
    SAVED_MODELS_DIR = "../saved_models/"
    RESULTS_DIR      = "../results/"

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Print actual folder names found so you can verify they match CLASS_NAMES
_train_path = os.path.join(DATASET_PATH, "Training")
_test_path  = os.path.join(DATASET_PATH, "Testing")
_train_cls  = sorted([d for d in os.listdir(_train_path) if os.path.isdir(os.path.join(_train_path, d))])
_test_cls   = sorted([d for d in os.listdir(_test_path)  if os.path.isdir(os.path.join(_test_path,  d))])
print(f"  Dataset path     : {DATASET_PATH}")
print(f"  Training folders : {_train_cls}")
print(f"  Testing  folders : {_test_cls}")
for _c in _train_cls:
    _imgs = [f for f in os.listdir(os.path.join(_train_path, _c)) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f"    Training/{_c}: {len(_imgs)} images")
print("Environment ready")


## Section 1: Imports & Configuration

In [ ]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
import joblib
from tqdm import tqdm
print("✓ Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME    = "07_TraditionalML_Ensemble"

DATASET_PATH     = globals().get("DATASET_PATH", "../MRI_DATASET/")
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

# Resize to 64×64 for manageable feature vector (12288 features)
IMG_HEIGHT       = 64
IMG_WIDTH        = 64
CHANNELS         = 3

RANDOM_SEED      = 42
SAVED_MODELS_DIR = globals().get("SAVED_MODELS_DIR", "../saved_models/")
RESULTS_DIR      = globals().get("RESULTS_DIR", "../results/")
RESULTS_NB_DIR   = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
HF_TOKEN         = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_NB_DIR, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("✓ Constants configured")
print(f"  Image size for features : {IMG_HEIGHT}×{IMG_WIDTH}×{CHANNELS}")
print(f"  Feature vector size     : {IMG_HEIGHT * IMG_WIDTH * CHANNELS}")
print(f"  Classes                 : {CLASS_NAMES}")

## Section 3: Data Loading & Verification

In [ ]:
def load_images_from_dir(base_dir, class_names, img_h, img_w):
    images, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_path = os.path.join(base_dir, cls)
        if not os.path.exists(cls_path):
            print(f"  WARNING: {cls_path} not found")
            continue
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"  {cls}: {len(files)} images")
        for fname in tqdm(files, desc=cls, leave=False):
            img = cv2.imread(os.path.join(cls_path, fname))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_w, img_h))
            images.append(img)
            labels.append(idx)
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)

print("Loading training images...")
X_train_raw, y_train = load_images_from_dir(TRAIN_DIR, CLASS_NAMES, IMG_HEIGHT, IMG_WIDTH)
print(f"✓ Train: {X_train_raw.shape}")

print("\nLoading test images...")
X_test_raw, y_test = load_images_from_dir(TEST_DIR, CLASS_NAMES, IMG_HEIGHT, IMG_WIDTH)
print(f"✓ Test : {X_test_raw.shape}")

print("\n" + "=" * 50)
print("DATA VERIFICATION")
print(f"Class order   : {CLASS_NAMES}")
print(f"Train samples : {X_train_raw.shape[0]}")
print(f"Test samples  : {X_test_raw.shape[0]}")
for u, c in zip(*np.unique(y_train, return_counts=True)):
    print(f"  {CLASS_NAMES[u]}: {c} training images")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
# Normalise to [0, 1] then flatten to 1D feature vectors
X_train = (X_train_raw / 255.0).reshape(X_train_raw.shape[0], -1)
X_test  = (X_test_raw  / 255.0).reshape(X_test_raw.shape[0],  -1)
n_features = X_train.shape[1]

print(f"✓ Normalised to [0, 1] and flattened")
print(f"  Train shape : {X_train.shape}")
print(f"  Test shape  : {X_test.shape}")
print(f"  Features    : {n_features}")

## Section 5: Model Definition

In [ ]:
rf  = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
dt  = DecisionTreeClassifier(random_state=RANDOM_SEED)
svm = SVC(probability=True, random_state=RANDOM_SEED, kernel='rbf', C=1.0)
lr  = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=0.1, n_jobs=-1)
ensemble_clf = VotingClassifier(
    estimators=[('rf', rf), ('dt', dt), ('svm', svm)],
    voting='soft',
    n_jobs=-1
)
print("✓ Classifiers: RF, DT, SVM, Logistic Regression, Soft-Vote Ensemble")

## Section 6: Model Training

In [ ]:
print("Training Random Forest...")
rf.fit(X_train, y_train)
print("✓ Random Forest trained")

print("Training Decision Tree...")
dt.fit(X_train, y_train)
print("✓ Decision Tree trained")

print("Training SVM (may take several minutes)...")
svm.fit(X_train, y_train)
print("✓ SVM trained")

print("Training Logistic Regression...")
lr.fit(X_train, y_train)
print("✓ Logistic Regression trained")

print("Training Soft-Vote Ensemble...")
ensemble_clf.fit(X_train, y_train)
print("✓ Ensemble trained")

## Section 7: Model Evaluation

In [ ]:
import re as _re

def evaluate_sklearn_model(clf, X_test, y_test, model_name="Model"):
    """Standard evaluation for sklearn classifiers."""
    _fname = _re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    y_pred       = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} — Confusion Matrix (Acc: {acc:.4f})')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_confusion_matrix.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n{model_name} — Classification Report")
    print("=" * 60)
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

    # ROC Curve
    y_true_bin = label_binarize(y_test, classes=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} — ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_roc_curve.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    # Precision-Recall Curve
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(recall, precision, label=f'{cls} (AP = {ap:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} — Precision-Recall Curve')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_NB_DIR, f"{_fname}_pr_curve.jpg"),
                dpi=150, bbox_inches='tight')
    plt.show()

    return acc, y_pred, y_pred_proba

rf_acc, rf_pred, rf_proba = evaluate_sklearn_model(rf, X_test_flat, y_test, "Traditional ML + Random Forest")
dt_acc, dt_pred, dt_proba = evaluate_sklearn_model(dt, X_test_flat, y_test, "Traditional ML + Decision Tree")
svm_acc, svm_pred, svm_proba = evaluate_sklearn_model(svm, X_test_flat, y_test, "Traditional ML + SVM")
ensemble_acc, ensemble_pred, ensemble_proba = evaluate_sklearn_model(ensemble_clf, X_test_flat, y_test, "Traditional ML + Soft-Vote Ensemble")

print("\n" + "=" * 50)
print("ACCURACY COMPARISON")
print(f"  Traditional ML + Random Forest  : {rf_acc:.4f}")
print(f"  Traditional ML + Decision Tree  : {dt_acc:.4f}")
print(f"  Traditional ML + SVM            : {svm_acc:.4f}")
print(f"  Traditional ML + Ensemble       : {ens_acc:.4f}")
print("=" * 50)

## Section 8: Save Model

In [ ]:
joblib.dump(rf,  SAVED_MODELS_DIR + 'traditional_rf.pkl')
joblib.dump(dt,  SAVED_MODELS_DIR + 'traditional_dt.pkl')
joblib.dump(svm, SAVED_MODELS_DIR + 'traditional_svm.pkl')
joblib.dump(lr,  SAVED_MODELS_DIR + 'traditional_lr.pkl')
joblib.dump(ensemble_clf, SAVED_MODELS_DIR + 'traditional_ensemble_model.pkl')
print(f"✓ All models saved to {SAVED_MODELS_DIR}")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Dataset  : {X_train.shape[0]} training + {X_test.shape[0]} test images")
print(f"Classes  : {CLASS_NAMES}")
print(f"Image size: {IMG_HEIGHT}×{IMG_WIDTH}×{CHANNELS} → {n_features} features (flattened)")
print(f"Seed     : {RANDOM_SEED}")
print("-" * 60)
print(f"  Random Forest       : {rf_acc:.4f}")
print(f"  Decision Tree       : {dt_acc:.4f}")
print(f"  SVM                 : {svm_acc:.4f}")
print(f"  Logistic Regression : {lr_acc:.4f}")
print(f"  Soft-Vote Ensemble  : {ens_acc:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)

## Section 10: HuggingFace Upload

Uploads the model file(s) saved in Section 8 and all result JPGs from Section 7 to `shehank98/brain-tumor-mri-models` on HuggingFace Hub.

Requires `HF_TOKEN` to be set (via Colab Secrets in Section 0, or `HF_TOKEN` env var).

In [ ]:
# ── HuggingFace repository ────────────────────────────────────────────────────
HF_REPO_ID = "shehank98/brain-tumor-mri-models"   # your HF repo

try:
    from huggingface_hub import HfApi, login as hf_login
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'])
    from huggingface_hub import HfApi, login as hf_login

if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set — skipping HuggingFace upload.")
    print("  Set it in Colab Secrets (key: HF_TOKEN) or as an env var.")
else:
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    api = HfApi()

    # Create repo if it does not exist yet
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model",
                    private=False, exist_ok=True)
    print(f"✓ Repository ready: https://huggingface.co/{HF_REPO_ID}")

    # Upload model files
    _model_files = ['traditional_rf.pkl', 'traditional_dt.pkl', 'traditional_svm.pkl', 'traditional_lr.pkl', 'traditional_ensemble_model.pkl']
    for _fname in _model_files:
        _local = os.path.join(SAVED_MODELS_DIR, _fname)
        if not os.path.exists(_local):
            print(f"  SKIP (not found): {_fname}")
            continue
        _size = os.path.getsize(_local) / 1e6
        print(f"  Uploading {_fname} ({_size:.1f} MB)...", end="", flush=True)
        api.upload_file(
            path_or_fileobj=_local,
            path_in_repo=f"models/{_fname}",
            repo_id=HF_REPO_ID,
            commit_message=f"Upload {_fname} from 07_TraditionalML_Ensemble",
        )
        print(" done")

    # Upload results JPGs
    _results_nb = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
    if os.path.exists(_results_nb):
        _jpgs = [f for f in os.listdir(_results_nb) if f.endswith('.jpg')]
        for _jpg in sorted(_jpgs):
            print(f"  Uploading result chart {_jpg}...", end="", flush=True)
            api.upload_file(
                path_or_fileobj=os.path.join(_results_nb, _jpg),
                path_in_repo=f"results/{NOTEBOOK_NAME}/{_jpg}",
                repo_id=HF_REPO_ID,
                commit_message=f"Add result chart {_jpg}",
            )
            print(" done")
        print(f"✓ {len(_jpgs)} result charts uploaded")
    else:
        print("  No result charts found — run Section 7 first")

    print(f"\n✓ Upload complete: https://huggingface.co/{HF_REPO_ID}")